In [1]:
# ============ CONCAT: FR clean + EN translation (DEV) ============
import os, re, json
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
    raise FileNotFoundError()
CACHE_DIR = find_cache_dir()
RESULTS_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'results')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
print(f"Device: {device}")

collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))

e5_name = 'intfloat/multilingual-e5-large'
tok = AutoTokenizer.from_pretrained(e5_name)
model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def average_pool(h, m):
    h = h.masked_fill(~m[..., None].bool(), 0.0)
    return h.sum(dim=1) / m.sum(dim=1)[..., None]

def clean_tweet(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    t = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def embed(texts, max_len=160):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    ds = ds.map(lambda e: tok(e["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt"),
                batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for b in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embed"):
            b = {k: v.to(device) for k, v in b.items()}
            o = model(**b)
            embs.append(average_pool(o.last_hidden_state, b['attention_mask']).cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def mrr_at_5(cs, lab):
    top5 = np.argsort(-cs, axis=1)[:, :5]
    keys = collection_keys[top5]
    m = (keys == lab[:, None])
    pos = np.argmax(m, axis=1)
    pos[~m.any(axis=1)] = -1
    rr = np.zeros_like(pos, dtype=float)
    rr[pos >= 0] = 1.0 / (pos[pos >= 0] + 1)
    return float(rr.mean())

# Load FR raw + cached translations
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "fr")
dev_raw = data["dev"]
dev_labels = np.array(dev_raw["pubkey"])

with open(os.path.join(CACHE_DIR, 'dev_en_fr.json'), encoding='utf-8') as f:
    dev_en = json.load(f)

assert len(dev_en) == len(dev_raw), f"len mismatch: {len(dev_en)} vs {len(dev_raw)}"

# Build concat queries
dev_concat = [f"{clean_tweet(orig)} {en}" for orig, en in zip(dev_raw["text"], dev_en)]

# Show samples
for i in range(3):
    print(f"[{i}] CONCAT: {dev_concat[i][:200]}\n")

# Embed + score
emb = embed(dev_concat)
score = mrr_at_5((emb @ collection_embeddings.T).numpy(), dev_labels)

print(f"\n=== FR DEV — concat (clean + EN trans) ===")
print(f"MRR@5: {score:.4f}")
print(f"Baseline:  0.4584")
print(f"Cleaning:  0.4620")
print(f"Pivot:     0.4618")
print(f"Δ vs cleaning: {score - 0.4620:+.4f}")

torch.save(emb, os.path.join(CACHE_DIR, 'dev_embeddings_fr_concat.pt'))
log = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp9_concat_fr_dev",
    "transformation": "clean_fr + en_translation concat",
    "language": "fr",
    "mrr5_dev": score,
    "delta_vs_cleaning": score - 0.4620,
}
with open(os.path.join(RESULTS_DIR, 'results.jsonl'), 'a') as f:
    f.write(json.dumps(log) + '\n')
print("Logged.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7044.16it/s]


E5 loaded.
[0] CONCAT: Psychose sévère provoquée par une inflammation cérébrale des récepteurs anti-N-méthyl D-aspartate suite à fpizerisation : minotte de 20 ans Severe psychosis caused by inflammation of the anti-N-methyl

[1] CONCAT: L Indomethacin (anti-inflammatoire non stéroïdien) employé conjointement au plan de traitement habituel chez les patients hospitalisés de COVID-19 a montré un atténuation marquée des symptômes et une 

[2] CONCAT: splendides photos : Étude sur la pollution des masques, analysée par des experts -à présent en ligne: La pollution des masques par des bactéries et champignons est une réalité. La toute initiale métho



Embed: 100%|██████████| 44/44 [00:42<00:00,  1.04it/s]



=== FR DEV — concat (clean + EN trans) ===
MRR@5: 0.4864
Baseline:  0.4584
Cleaning:  0.4620
Pivot:     0.4618
Δ vs cleaning: +0.0244
Logged.
